# ECE/CS 281b Final Project — Preprocessing Notebook

Constructs the evaluation dataset used in `experiment.ipynb`.  
Samples a stratified 5 000-image subset from the ImageNet validation set,  
then applies five human-vision-motivated degradations at five severity levels  
to produce 25 preprocessed tensor files.

All tensors are saved as raw \[0, 1\] `float16` — ImageNet normalisation is deferred  
to the experiment notebook so that degradation functions operate in pixel space.

ImageNet-C results are sourced directly from the RobustBench leaderboard;  

**Output layout:**
```
preprocessed/
├── clean.pt         # 5 000 clean images, float16 [0, 1], shape (5000, 3, 224, 224)
├── labels.pt        # ground-truth ImageNet class indices, shape (5000,)
└── hvm_d/           # 5 degradations × 5 severities = 25 .pt files
    ├── foveated_blur_sev{1..5}.pt
    ├── contrast_loss_sev{1..5}.pt
    ├── shot_noise_sev{1..5}.pt
    ├── fixation_jitter_sev{1..5}.pt
    └── composed_sev{1..5}.pt
```


---
## Step 0 — Imports & Configuration

Imports all dependencies, sets the global random seed for reproducibility,  
and defines directory paths and dataset parameters.

**Parameters to configure before running:**
- `IMAGENET_VAL_DIR` — path to the local ImageNet validation split
- `SAMPLES_PER_CLASS` — images sampled per class (default: 5, yielding N = 5 000)

In [ ]:
import random, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
from scipy.ndimage import gaussian_filter
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Reproducibility
# A single global seed governs Python, NumPy, and PyTorch RNGs, ensuring that
# the stratified subset selection and per-image degradation seeds are identical
# across independent runs.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths
IMAGENET_VAL_DIR  = "dataset/imagenet/val"
HVM_D_DIR         = Path("preprocessed/hvm_d")
HVM_D_DIR.mkdir(parents=True, exist_ok=True)
Path("preprocessed").mkdir(exist_ok=True)

# Dataset parameters
SAMPLES_PER_CLASS = 5                   # 5 images x 1 000 classes = 5 000 total
SEVERITIES        = list(range(1, 6))   # severity levels {1, 2, 3, 4, 5}

print(f"HVM_D output dir : {HVM_D_DIR}")
print(f"Total images     : {SAMPLES_PER_CLASS * 1000}")

---
## Step 1 — Load Stratified Subset

**`make_stratified_subset`** samples exactly `samples_per_class` images per ImageNet class  
using `random.Random(seed)` with a fixed seed, so the **same 5 000 images are loaded  
every time the kernel restarts** — essential for consistent resume-after-crash behaviour.

> **Normalization is intentionally deferred to the experiment notebook.**  
> `base_transform` outputs raw [0, 1] float32 tensors (Resize → CenterCrop → ToTensor).  
> All noise functions assume pixel values are in [0, 1]  
> (e.g. the Poisson scale parameter in `shot_noise` depends on this range).  
> The saved `.pt` files are therefore raw [0, 1] — no ImageNet normalization applied.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Resize to 256 then centre-crop to 224, matching standard ImageNet eval preprocessing.
# ToTensor() converts PIL images to float32 tensors in [0, 1].
base_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),  
])

def make_stratified_subset(root: str, samples_per_class: int = 10, seed: int = 42) -> Subset:
    """Return a stratified Subset of an ImageFolder dataset.

    Samples exactly `samples_per_class` images per class using a seeded RNG,
    guaranteeing that the same indices are selected on every run regardless of
    filesystem ordering or kernel restarts.

    Args:
        root:               Path to an ImageFolder-compatible directory.
        samples_per_class:  Number of images to sample per class.
        seed:               Random seed for reproducible index selection.

    Returns:
        A torch.utils.data.Subset containing samples_per_class x n_classes images.
    """
    dataset = ImageFolder(root, transform=base_transform)
    rng     = random.Random(seed)
    class_to_indices = defaultdict(list)
    for idx, (_, label) in enumerate(dataset.samples):
        class_to_indices[label].append(idx)
    selected = []
    for label in sorted(class_to_indices):
        pool   = class_to_indices[label]
        chosen = rng.sample(pool, min(samples_per_class, len(pool)))
        selected.extend(chosen)
    print(f"Subset: {len(selected)} images ({samples_per_class} per class)")
    return Subset(dataset, selected)

val_subset = make_stratified_subset(IMAGENET_VAL_DIR, SAMPLES_PER_CLASS, SEED)

# Load entire subset into CPU memory
# sweep (Step 3), which applies 25 transforms to the same base images.
print("Loading clean images into memory...")
all_imgs, all_labels = [], []
loader = DataLoader(val_subset, batch_size=256, shuffle=False, num_workers=4)
for imgs, labels in tqdm(loader):
    all_imgs.append(imgs)
    all_labels.append(labels)
all_imgs   = torch.cat(all_imgs)    # (5000, 3, 224, 224) float32 [0, 1]
all_labels = torch.cat(all_labels)  # (5000,) int64

# Save clean images as float16
clean_path = Path("preprocessed/clean.pt")
torch.save(all_imgs.half(), clean_path)
print(f"Saved -> {clean_path}  ({clean_path.stat().st_size/1e9:.2f} GB)")

# Labels are stored as int64 (no precision loss); saved once and shared across all
torch.save(all_labels, "preprocessed/labels.pt")
print(f"Images shape : {all_imgs.shape}")
print(f"Labels saved -> preprocessed/labels.pt")


---
## Step 2 — Human-Vision-Motivated Degradation Functions

Four human-vision-motivated degradations plus a composed pipeline.  
Every function has the signature `(img_t: Tensor, severity: int, rng=None) → Tensor`  
and operates entirely in **[0, 1] pixel space** before normalization.

**`foveated_blur`** — eccentricity-dependent blur (Geisler & Perry, 1998).  
The blur magnitude at each pixel scales linearly with its Euclidean distance from  
the image centre: `σ(p) = (d(p) / d_max) × (severity × 2)`, where `d_max` is the  
centre-to-corner distance. Spatially-varying convolution is approximated by blending  
K=7 `scipy.ndimage.gaussian_filter` passes at discrete σ levels via normalised  
Gaussian weights. Centre pixels remain sharp (σ=0); corner pixels reach σ=2×severity.

**`contrast_loss`** — linear contrast compression toward mid-grey (0.5),  
modelling retinal ganglion cell gain control. Pure torch ops; fastest of the four.

**`shot_noise`** — signal-dependent Poisson noise via `np.random.default_rng().poisson()`.  
Darker pixels receive proportionally more noise, matching photoreceptor quantum noise.  
Unlike ImageNet-C shot noise (signal-independent), this is physically grounded.

**`fixation_jitter`** — random integer-pixel translation modelling microsaccades  
(Martinez-Conde et al., 2004). Uses `np.pad(mode='edge')` + crop for border replication.

**`composed_pipeline`** — applies all four in physiological order:  
`foveated_blur → contrast_loss → shot_noise → fixation_jitter`

In [ ]:
def foveated_blur(img_t: torch.Tensor, severity: int, rng=None) -> torch.Tensor:
    img_np = img_t.numpy().transpose(1, 2, 0)
    H, W   = img_np.shape[:2]
    cx, cy = W / 2.0, H / 2.0
    y, x   = np.ogrid[:H, :W]
    dist   = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_d  = np.sqrt(cx**2 + cy**2)
    sigma_map = (dist / max_d) * (severity * 2.0)
    n_levels = 7
    sigmas   = np.linspace(0, severity * 2.0, n_levels)
    out      = np.zeros_like(img_np, dtype=np.float32)
    weights  = np.zeros((H, W), dtype=np.float32)
    bw       = max(severity * 2.0 / n_levels, 0.5)
    for s in sigmas:
        blurred = gaussian_filter(img_np.astype(np.float32), sigma=[s, s, 0])
        w       = np.exp(-0.5 * ((sigma_map - s) / bw)**2).astype(np.float32)
        out    += blurred * w[:, :, None]
        weights += w
    out = np.clip(out / (weights[:, :, None] + 1e-8), 0, 1)
    return torch.from_numpy(out.transpose(2, 0, 1).copy())

def contrast_loss(img_t: torch.Tensor, severity: int, rng=None) -> torch.Tensor:
    factors = [0.93, 0.80, 0.65, 0.50, 0.38]
    f = factors[severity - 1]
    return torch.clamp(0.5 + (img_t - 0.5) * f, 0, 1)

def shot_noise(img_t: torch.Tensor, severity: int, rng=None) -> torch.Tensor:
    scales = [5000, 1000, 300, 100, 50]
    scale  = scales[severity - 1]
    _rng   = np.random.default_rng(rng)
    img_np = img_t.numpy()
    noisy  = _rng.poisson(img_np * scale).astype(np.float32) / scale
    return torch.from_numpy(np.clip(noisy, 0, 1))

def fixation_jitter(img_t: torch.Tensor, severity: int, rng=None) -> torch.Tensor:
    """
    Random integer pixel shift with border (replication) padding.
    """
    _rng = np.random.default_rng(rng)
    dx   = int(_rng.integers(-severity, severity + 1))
    dy   = int(_rng.integers(-severity, severity + 1))

    img_np = img_t.numpy().transpose(1, 2, 0)   # (H, W, C)
    H, W   = img_np.shape[:2]

    # Pad with border replication, then crop — equivalent to padding_mode='border'
    pad_h, pad_w = abs(dy), abs(dx)
    img_pad = np.pad(img_np,
                     ((pad_h, pad_h), (pad_w, pad_w), (0, 0)),
                     mode='edge')   # 'edge' = replication padding

    # Compute crop origin based on shift direction
    y0 = pad_h - dy   # dy>0 → shift down → crop higher up
    x0 = pad_w - dx
    cropped = img_pad[y0:y0+H, x0:x0+W, :]

    return torch.from_numpy(cropped.astype(np.float32).transpose(2, 0, 1).copy())

def composed_pipeline(img_t: torch.Tensor, severity: int, rng=None) -> torch.Tensor:
    _rng = np.random.default_rng(rng)
    img  = foveated_blur(img_t, severity, _rng.integers(0, 2**31))
    img  = contrast_loss(img,   severity)
    img  = shot_noise(img,      severity, _rng.integers(0, 2**31))
    img  = fixation_jitter(img, severity, _rng.integers(0, 2**31))
    return img

DEGRADATIONS = {
    "foveated_blur"  : foveated_blur,
    "contrast_loss"  : contrast_loss,
    "shot_noise"     : shot_noise,
    "fixation_jitter": fixation_jitter,
    "composed"       : composed_pipeline,
}
print("Degradation functions defined:", list(DEGRADATIONS.keys()))


---
## Step 3 — Preprocess Human-Vision-Motivated Degradations

Processes all 5 degradations × 5 severities = **25 combinations**.  
Each combination is saved as a single `(5000, 3, 224, 224)` float16 `.pt` file.

**Files are raw [0, 1] float16 — no normalization applied.**  
The experiment notebook applies `torchvision.transforms.Normalize` just before model inference.


In [ ]:
# N is used throughout this cell as the total number of images to process.
# Each output file will have shape (N, 3, 224, 224) = (5000, 3, 224, 224).
N = len(all_imgs)

In [ ]:
total_hvm_d = len(DEGRADATIONS) * len(SEVERITIES)
CHUNK = 500   # images processed per iteration

print(f"Generating {total_hvm_d} degraded tensor files (N={N} images each)...\n")

for deg_name, deg_fn in DEGRADATIONS.items():
    for sev in SEVERITIES:
        out_path = HVM_D_DIR / f"{deg_name}_sev{sev}.pt"
        if out_path.exists():
            print(f"  skip (exists): {out_path.name}")
            continue

        chunks  = []
        n_chunks = (N + CHUNK - 1) // CHUNK
        for start in tqdm(range(0, N, CHUNK),
                          desc=f"  {deg_name} sev={sev}",
                          total=n_chunks, unit="chunk", ncols=80):
            end   = min(start + CHUNK, N)
            # Per-image seed = SEED + i ensures independent, reproducible stochasticity
            batch = torch.stack([
                deg_fn(all_imgs[i], sev, rng=SEED + i)
                for i in range(start, end)
            ])
            chunks.append(batch.half())  # cast to float16 to halve storage
            del batch

        degraded = torch.cat(chunks)  # (N, 3, 224, 224) float16, raw [0, 1]
        torch.save(degraded, out_path)
        del degraded, chunks
        print(f"  Saved -> {out_path.name}  ({out_path.stat().st_size/1e9:.2f} GB)")

print("\nHuman-vision-motivated degradation preprocessing complete.")


---
## Step 4 — Verify Output

Checks file count, total disk usage, tensor shape, dtype, and value range.

**Expected results:**
- human-vision-motivated degradations: **25 files**
- Shape: `(5000, 3, 224, 224)`, dtype `float16`
- Value range: `[0.0, 1.0]` (raw, un-normalized)


In [ ]:
hvm_d_files = list(HVM_D_DIR.glob("*.pt"))
hvm_d_gb    = sum(f.stat().st_size for f in hvm_d_files) / 1e9

print(f"Human-vision-motivated degradations files : {len(hvm_d_files):3d}  ({hvm_d_gb:.1f} GB)")
print(f"Labels file     : preprocessed/labels.pt")

# Quick shape / dtype / range check
sample = torch.load(HVM_D_DIR / "foveated_blur_sev1.pt", map_location="cpu")
print(f"\nSample tensor shape : {sample.shape}")
print(f"dtype               : {sample.dtype}")
print(f"value range         : [{sample.float().min():.3f}, {sample.float().max():.3f}]")
assert sample.shape == (len(all_labels), 3, 224, 224), "Shape mismatch!"
assert sample.dtype == torch.float16
print("\n All checks passed.")


---
## Step 5 — Visual Preview

Renders all five degradations at `severity=3` for image index 0 as a sanity check.  
Tensors are stored as raw \[0, 1\] float16, so no denormalisation is required before display.  
The figure is saved to `results/preview_hvm_d_sev3.png` for inclusion in the report.


In [ ]:
import torch
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
HVM_D_DIR = Path("preprocessed/hvm_d")

SEV     = 3
IMG_IDX = 0

HVM_DEGS = [
    "foveated_blur", "contrast_loss", "shot_noise",
    "fixation_jitter", "composed",
]

entries = []   # (title, img_array)

for deg in HVM_DEGS:
    t   = torch.load(HVM_D_DIR / f"{deg}_sev{SEV}.pt", map_location="cpu")[IMG_IDX]
    img = t.float().permute(1, 2, 0).clamp(0, 1).numpy()   # raw [0,1], no denorm needed
    entries.append((deg.replace("_", "\n"), img))

fig, axes = plt.subplots(1, 5, figsize=(5 * 2.8, 3.2))
fig.suptitle(f"Human-Vision-Motivated Degradations — image #{IMG_IDX}, severity={SEV}", fontsize=15, y=1.02)

for ax, (title, img) in zip(axes, entries):
    ax.imshow(img)
    ax.set_title(title, fontsize=12, color="#4e79a7")
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor("#4e79a7")
        spine.set_linewidth(2)

plt.tight_layout()
Path("results").mkdir(exist_ok=True)
plt.savefig("results/preview_hvm_d_sev3.png", dpi=330, bbox_inches="tight")
plt.show()
print("Saved → results/preview_hvm_d_sev3.png")